# Sakura-GalTransl-14B-v3.8 — Colab L4 一鍵自動化翻譯

**使用步驟：**
1. 確認右上角硬體加速器為 **L4 GPU**（24 GB VRAM；若配到 T4 會自動拒跑並釋放資源）。
2. 確認本機 Agent 已透過 `drive_bridge.py push` 將 `<job>.jsonl` 與 `.manifest.json` 推入 `MyDrive/galgame_mtl/sakura/input/`。
3. 按 **`Ctrl+F9`（全部執行）**，點「允許」授權 Google Drive 後即可離開電腦。
4. 翻譯完成（或遇錯停止）後，`finally` 區塊會自動保存日誌、`flush_and_unmount()` 同步雲端，並執行 `runtime.unassign()` 釋放 L4 停止計費。

In [ ]:
#@title 1. 執行參數 { display-mode: "form" }
AUTO_DISCONNECT = True #@param {type:"boolean"}
TW_CONVERT_MODE = "s2t" #@param ["s2t", "s2twp"]
FORCE_RERUN = False #@param {type:"boolean"}
REBUILD_LLAMA_BIN = False #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/galgame_mtl" #@param {type:"string"}
REPO_URL = "https://github.com/poynt2005/hanhua-mtl.git" #@param {type:"string"}
BRANCH = "sakura-ollama-colab" #@param {type:"string"}

In [ ]:
#@title 2. 一鍵啟動 GPU 4 道驗活 + 4 路並行翻譯 + QA + 自動斷線
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import drive, runtime

def _load_runner_dir(drive_root: Path) -> Path:
    drive_scripts = drive_root / "scripts" / "sakura"
    if (drive_scripts / "runner.py").is_file() and (drive_scripts / "schema.py").is_file():
        local_scripts = Path("/content/_runner_sakura")
        local_scripts.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_scripts / "schema.py", local_scripts / "schema.py")
        shutil.copy2(drive_scripts / "runner.py", local_scripts / "runner.py")
        print(f"[init] 已從 Drive 載入腳本: {drive_scripts}")
        return local_scripts

    repo_dir = Path("/content/_repo")
    if repo_dir.exists():
        shutil.rmtree(repo_dir, ignore_errors=True)
    print(f"[init] Drive 無同步腳本，改由 Git 取得 {BRANCH}...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)],
        check=True,
    )
    return repo_dir / "deploy" / "colab-sakura"

mounted = False
rc = 1
try:
    drive.mount("/content/drive", force_remount=False)
    mounted = True
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "opencc-python-reimplemented"],
        check=True,
    )
    runner_dir = _load_runner_dir(Path(DRIVE_ROOT))
    cmd = [
        sys.executable,
        "-u",
        str(runner_dir / "runner.py"),
        "--drive-root", DRIVE_ROOT,
        "--tw-mode", TW_CONVERT_MODE,
    ]
    if FORCE_RERUN:
        cmd.append("--force")
    if REBUILD_LLAMA_BIN:
        cmd.append("--rebuild-bin")
    proc = subprocess.run(cmd)
    rc = proc.returncode
    if rc != 0:
        raise RuntimeError(f"runner.py 結束碼 {rc}，請查看 work/llama.log 與 failures.jsonl")
    print("\n[DONE] 全部翻譯任務已完成並寫入 .DONE.json！")
finally:
    if mounted:
        try:
            print("[flush] 正在將輸出與日誌同步回 Google Drive...")
            drive.flush_and_unmount()
            print("[flush] Drive 同步完成。")
        except Exception as exc:
            print(f"[flush] warn: {exc}")
    if AUTO_DISCONNECT:
        print("[power] AUTO_DISCONNECT=True -> 正在釋放 Colab GPU 執行階段...")
        runtime.unassign()
